# CardioFusion-XAI — preprocessing artifact validation

The original preprocessing notebooks remain untouched. This additional notebook verifies their outputs and expected tensor/manifest contracts before training. It does not rerun preprocessing or overwrite processed datasets.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

def find_root(start=None):
    current=Path(start or Path.cwd()).resolve()
    for p in [current,*current.parents]:
        if (p/"src").is_dir() and (p/"config").is_dir(): return p
        if (p/"ml-service"/"src").is_dir(): return p/"ml-service"
    raise FileNotFoundError("Could not locate ml-service.")
ML_ROOT=find_root()
DATA=ML_ROOT/"data"/"processed"
checks=[]

## 1. Biomarker raw source schema readiness

The clinical source loader is used only for canonical schema inspection. Fitted imputation and scaling remain in the FT-Transformer training notebook and are fitted on the training split only.

In [2]:
from src.training.biomarker_training_data import load_biomarker_source
raw_root=ML_ROOT/"data"/"raw"/"biomarkers"
for source in ["zheen","framingham","uci_heart_failure","mi_complications"]:
    try:
        item=load_biomarker_source(raw_root,source)
        checks.append({"modality":"clinical_biomarkers","source":source,"status":"ready",
                       "rows":len(item.features),"features":item.features.shape[1],"targets":len(item.targets)})
    except Exception as exc:
        checks.append({"modality":"clinical_biomarkers","source":source,"status":"not_ready","reason":f"{type(exc).__name__}: {exc}"})

## 2. ECG processed manifest and waveform shape check

In [3]:
from pathlib import Path
ptb_manifest=DATA/"ecg"/"ptbxl"/"100hz_record_zscore"/"manifests"/"ptbxl_manifest.csv"
if ptb_manifest.is_file():
    frame=pd.read_csv(ptb_manifest)
    needed={"processed_path","patient_id","ecg_id","split"}
    missing=needed-set(frame.columns)
    if missing: raise ValueError(f"PTB-XL manifest missing columns: {sorted(missing)}")
    sample=frame.iloc[0]
    waveform_path=ML_ROOT/str(sample["processed_path"])
    if not waveform_path.is_file(): raise FileNotFoundError(f"Manifest references missing waveform: {waveform_path}")
    waveform=np.load(waveform_path,allow_pickle=False)
    if waveform.shape not in {(1000,12),(12,1000)}: raise ValueError(f"Unexpected ECG waveform shape: {waveform.shape}")
    checks.append({"modality":"ecg","source":"ptbxl","status":"ready","rows":len(frame),"sample_shape":str(waveform.shape)})
else:
    checks.append({"modality":"ecg","source":"ptbxl","status":"missing_manifest","expected":str(ptb_manifest)})

## 3. EchoNet manifest/file layout check

In [4]:
echo_candidates=[DATA/"echo"/"echonet"/"echonet_panecho_manifest.csv", ML_ROOT/"data"/"raw"/"echo"/"echonet"/"FileList.csv"]
echo_file=next((p for p in echo_candidates if p.is_file()),None)
if echo_file:
    echo_df=pd.read_csv(echo_file)
    checks.append({"modality":"echo","source":"echonet_dynamic","status":"metadata_ready","rows":len(echo_df),"columns":len(echo_df.columns),"path":str(echo_file)})
else:
    checks.append({"modality":"echo","source":"echonet_dynamic","status":"missing_manifest_or_filelist","expected":"; ".join(map(str,echo_candidates))})

## 4. Validation summary

In [5]:
check_df=pd.DataFrame(checks)
display(check_df)
out=ML_ROOT/"artifacts"/"preprocessing"/"preprocessing_validation.csv"
out.parent.mkdir(parents=True,exist_ok=True)
check_df.to_csv(out,index=False)
print("Validation inventory saved:",out)
print("This report verifies availability/contracts only; it does not claim model performance.")

,modality,source,status,rows,features,targets,sample_shape,columns,path
0,clinical_biomarkers,zheen,ready,1319,8.0,1.0,NaN,NaN,NaN
1,clinical_biomarkers,framingham,ready,4240,15.0,1.0,NaN,NaN,NaN
2,clinical_biomarkers,uci_heart_failure,ready,299,11.0,1.0,NaN,NaN,NaN
3,clinical_biomarkers,mi_complications,ready,1700,107.0,12.0,NaN,NaN,NaN
4,ecg,ptbxl,ready,21837,NaN,NaN,"(1000, 12)",NaN,NaN
5,echo,echonet_dynamic,metadata_ready,1530,NaN,NaN,NaN,7.0,D:\Programming\VS Code\Projects\CardioFusion-X...


Validation inventory saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\preprocessing\preprocessing_validation.csv
This report verifies availability/contracts only; it does not claim model performance.
